# Tahap 4 - Case Solution Reuse
## CBR Putusan Wanprestasi PN Surabaya

**Tujuan:** Gunakan putusan lama sebagai dasar pencarian solusi untuk kasus baru.

**Input:**
- `data/processed/cases.csv` → output Tahap 2
- Fungsi `retrieve()` & `vectorizer`/`tfidf_matrix` dari Tahap 3 (dibangun ulang di notebook ini agar dapat dijalankan independen)

**Output:**
- Struktur `{case_id: solusi_text}` (case_solutions)
- Fungsi `predict_outcome(query)` dengan 2 algoritma: **Majority Vote** & **Weighted Similarity**
- Demo manual 5 kasus baru
- `data/results/predictions.csv`

---

## Alur Tahap 4

| Cell | Fungsi |
|---|---|
| **Cell 1** | Import library |
| **Cell 2** | Load data & bangun ulang TF-IDF + `retrieve()` (dari Tahap 3) |
| **Cell 3** | Ekstrak Solusi → struktur `{case_id: solusi_text}` |
| **Cell 4** | Algoritma Prediksi: Majority Vote |
| **Cell 5** | Algoritma Prediksi: Weighted Similarity |
| **Cell 6** | Implementasi fungsi `predict_outcome()` |
| **Cell 7** | Demo Manual - 5 kasus baru |
| **Cell 8** | Simpan `data/results/predictions.csv` |
| **Cell 9** | Ringkasan akhir Tahap 4 |

## Setup - Jangkar Working Directory ke Root Repository

**Jalankan cell ini SELALU sebagai cell pertama**, sebelum cell lain di notebook ini.

Notebook ini disimpan di dalam folder `notebooks/`, tapi seluruh path data di notebook ini
(mis. `'data/processed/cases.csv'`) ditulis **relatif terhadap root repository**, bukan terhadap
folder `notebooks/`. Secara default, Jupyter menjalankan kernel dengan *working directory* = folder
tempat file `.ipynb` itu sendiri berada. Tanpa cell ini, path seperti `'data/raw'` akan dicari di
`notebooks/data/raw` (tidak ada) dan menyebabkan `FileNotFoundError`.

Cell ini **aman dijalankan berkali-kali** (idempotent) — begitu working directory sudah berada di
root repository (folder yang punya subfolder `data/`), cell ini tidak akan berpindah lagi.

## Cell 1 — Import Library

In [52]:
import os, re, json
import numpy as np
import pandas as pd
from typing import List, Dict
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Semua library siap!')

Semua library siap!


## Cell 2 — Load Data & Bangun Ulang TF-IDF + `retrieve()`

Tahap 4 membutuhkan fungsi `retrieve()` dari Tahap 3. Agar notebook ini dapat dijalankan
secara independen (tidak bergantung pada *kernel state* notebook Tahap 3), komponen
TF-IDF dan `retrieve()` dibangun ulang di sini dengan **konfigurasi & preprocessing yang
identik** dengan Tahap 3.

In [53]:
PATH_CASES_CSV     = 'data/processed/cases.csv'
PATH_PREDICTIONS    = 'data/results/predictions.csv'

os.makedirs('data/results', exist_ok=True)

# ── Load cases.csv (output Tahap 2) ──────────────────────────────────────────
df = pd.read_csv(PATH_CASES_CSV, encoding='utf-8-sig')
df = df[df['text_full'].notna() & (df['text_full'].str.len() > 100)].reset_index(drop=True)

# ── Filter ke label 0/1 saja — konsisten dengan df_model di Tahap 3 ──────────
# (dokumen Damai/Penetapan & Masih Proses sudah dihapus di Tahap 2,
#  filter ini sekadar jaring pengaman bila suatu saat ada label lain masuk lagi)
df = df[df['label_putusan'].isin([0, 1])].reset_index(drop=True)

print('=' * 62)
print('  KONFIGURASI TAHAP 4 — CASE SOLUTION REUSE')
print('=' * 62)
print(f'  Total kasus tersedia (case base) : {len(df)}')
print(f'  Label Dikabulkan      (1)        : {(df["label_putusan"]==1).sum()}')
print(f'  Label Ditolak/NO      (0)        : {(df["label_putusan"]==0).sum()}')
print('=' * 62)

  KONFIGURASI TAHAP 4 — CASE SOLUTION REUSE
  Total kasus tersedia (case base) : 54
  Label Dikabulkan      (1)        : 30
  Label Ditolak/NO      (0)        : 24


In [54]:
# ── Preprocessing query — identik dengan Tahap 3 ──────────────────────────────
STOPWORDS = {
    'dan', 'di', 'ke', 'dari', 'yang', 'ini', 'itu', 'atau',
    'pada', 'dengan', 'adalah', 'juga', 'oleh', 'serta', 'pula',
    'pun', 'nya', 'si', 'tidak', 'telah', 'akan', 'bahwa', 'untuk',
    'dalam', 'atas', 'tersebut', 'para', 'sebagai', 'sudah',
    'hal', 'maka', 'agar', 'karena', 'namun', 'tetapi', 'jika',
    'pihak', 'perkara', 'maupun', 'dapat', 'saat', 'telah', 'juga',
}

def preprocess_query(teks: str) -> str:
    """
    Preprocessing teks sebelum diubah ke vektor TF-IDF.
    1) Lowercase  2) Hapus tanda baca  3) Normalisasi spasi  4) Filter stopwords
    """
    teks = teks.lower()
    teks = re.sub(r'[^\w\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    tokens = [t for t in teks.split() if t not in STOPWORDS and len(t) >= 3]
    return ' '.join(tokens)


# ── Bangun ulang TF-IDF Vectorizer pada seluruh case base ────────────────────
corpus   = df['text_full'].apply(preprocess_query).tolist()
case_ids = df['case_id'].tolist()

vectorizer = TfidfVectorizer(
    max_features = 5000,
    min_df       = 2,
    max_df       = 0.95,
    ngram_range  = (1, 2),
    sublinear_tf = True,
)
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f'TF-IDF matrix dibangun ulang: {tfidf_matrix.shape[0]} dokumen x {tfidf_matrix.shape[1]:,} fitur')

TF-IDF matrix dibangun ulang: 54 dokumen x 5,000 fitur


In [55]:
def retrieve(query: str, k: int = 5) -> List[str]:
    """
    Temukan k kasus paling mirip dengan query kasus baru. (Identik Tahap 3)
    1) Pre-process query  2) Vektor query  3) Cosine-similarity  4) Top-k case_id
    """
    query_bersih = preprocess_query(query)
    query_vec    = vectorizer.transform([query_bersih])
    sim_scores   = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_k_idx    = np.argsort(sim_scores)[::-1][:k]
    return [case_ids[i] for i in top_k_idx]


def retrieve_with_scores(query: str, k: int = 5) -> List[dict]:
    """
    Versi retrieve() dengan skor similarity untuk keperluan weighted-prediction.
    """
    query_bersih = preprocess_query(query)
    query_vec    = vectorizer.transform([query_bersih])
    sim_scores   = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_k_idx    = np.argsort(sim_scores)[::-1][:k]

    hasil = []
    for i in top_k_idx:
        row = df.iloc[i]
        hasil.append({
            'case_id'    : case_ids[i],
            'no_perkara' : row['no_perkara'],
            'similarity' : round(float(sim_scores[i]), 4),
            'label'      : int(row['label_putusan']),
            'pihak'      : row['pihak'],
        })
    return hasil


print('Fungsi retrieve() & retrieve_with_scores() siap (rebuilt dari Tahap 3).')

Fungsi retrieve() & retrieve_with_scores() siap (rebuilt dari Tahap 3).


## Cell 3 — Ekstrak Solusi: `{case_id: solusi_text}`

Sesuai soal poin **d.i**: dari kasus top-k, ambil elemen putusan (amar putusan)
sebagai "solusi" atas kasus baru. Disimpan dalam struktur dict `case_solutions`.

In [56]:
def ringkas_amar(amar_lengkap: str, max_chars: int = 300) -> str:
    """
    Meringkas teks amar_lengkap menjadi solusi singkat yang mudah dibaca.
    Membuang kata berulang 'mengadili', 'dalam konpensi', dsb di awal jika
    sudah cukup jelas dari konteks, lalu memotong ke max_chars karakter.
    """
    if not isinstance(amar_lengkap, str) or len(amar_lengkap.strip()) == 0:
        return 'AMAR_TIDAK_TERSEDIA'
    teks = amar_lengkap.strip()
    if len(teks) > max_chars:
        teks = teks[:max_chars].rsplit(' ', 1)[0] + ' ...'
    return teks


LABEL_NAMA = {
    1  : 'Dikabulkan',
    0  : 'Ditolak/NO',
}

# ── Bangun struktur {case_id: solusi_text} ────────────────────────────────────
case_solutions: Dict[str, str] = {}
case_solutions_label: Dict[str, int] = {}   # label numerik per case_id (untuk majority vote)

for _, row in df.iterrows():
    case_id = row['case_id']
    amar_sumber = row['amar_lengkap'] if 'amar_lengkap' in df.columns and pd.notna(row.get('amar_lengkap')) else row.get('amar_putusan', '')
    case_solutions[case_id]       = ringkas_amar(amar_sumber)
    case_solutions_label[case_id] = int(row['label_putusan'])

print(f'Struktur case_solutions dibangun: {len(case_solutions)} entri')
print()
print('Contoh 3 entri pertama:')
for cid in list(case_solutions.keys())[:3]:
    lbl = LABEL_NAMA.get(case_solutions_label[cid], '?')
    print(f'  {cid} [{lbl}]')
    print(f'    -> {case_solutions[cid][:150]}...')
    print()

Struktur case_solutions dibangun: 54 entri

Contoh 3 entri pertama:
  case_001 [Dikabulkan]
    -> mengadili dalam konpensi dalam eksepsi menolak eksepsi dari tergugat dalam pokok perkara 1 mengabulkan gugatan penggugat sebagian 2 menyatakan terguga...

  case_002 [Ditolak/NO]
    -> mengadili dalam konpensi dalam eksepsi menerima eksepsi dari tergugat dalam pokok perkara menyatakan gugatan penggugat tidak dapat diterima niet ontva...

  case_003 [Dikabulkan]
    -> mengadili 1 menyatakan bahwa tergugat yang telah dipanggil dengan patut untuk datang menghadap dipersidangan tidak hadir 2 mengabulkan gugatan penggug...



## Cell 4 — Algoritma Prediksi 1: Majority Vote

Memilih solusi (label) yang **paling banyak muncul** di antara top-k kasus hasil retrieval.
Tie-break: jika ada lebih dari satu label dengan jumlah suara sama, pilih yang memiliki
rata-rata similarity tertinggi.

In [57]:
def majority_vote(top_k_hasil: List[dict]) -> dict:
    """
    Algoritma Majority Vote: pilih label yang paling banyak muncul di top-k.

    Args:
        top_k_hasil : List[dict] — hasil retrieve_with_scores()

    Returns:
        dict berisi: predicted_label, predicted_label_nama, vote_count, vote_detail
    """
    labels = [h['label'] for h in top_k_hasil]
    counter = Counter(labels)
    max_vote = max(counter.values())
    kandidat = [lbl for lbl, cnt in counter.items() if cnt == max_vote]

    if len(kandidat) == 1:
        label_terpilih = kandidat[0]
    else:
        # Tie-break: rata-rata similarity tertinggi di antara kandidat
        avg_sim = {}
        for lbl in kandidat:
            sims = [h['similarity'] for h in top_k_hasil if h['label'] == lbl]
            avg_sim[lbl] = sum(sims) / len(sims)
        label_terpilih = max(avg_sim, key=avg_sim.get)

    return {
        'predicted_label'      : label_terpilih,
        'predicted_label_nama' : LABEL_NAMA.get(label_terpilih, '?'),
        'vote_count'           : dict(counter),
        'metode'               : 'majority_vote',
    }


# ── Test cepat ──────────────────────────────────────────────────────────────
query_test = 'tergugat tidak membayar hutang sesuai perjanjian wanprestasi ganti rugi'
top5_test  = retrieve_with_scores(query_test, k=5)
hasil_mv   = majority_vote(top5_test)

print('Fungsi majority_vote() siap!')
print()
print(f'Query        : "{query_test}"')
print(f'Top-5 labels : {[h["label"] for h in top5_test]}')
print(f'Vote count   : {hasil_mv["vote_count"]}')
print(f'Prediksi     : {hasil_mv["predicted_label_nama"]} (label {hasil_mv["predicted_label"]})')

Fungsi majority_vote() siap!

Query        : "tergugat tidak membayar hutang sesuai perjanjian wanprestasi ganti rugi"
Top-5 labels : [0, 1, 1, 1, 1]
Vote count   : {0: 1, 1: 4}
Prediksi     : Dikabulkan (label 1)


## Cell 5 — Algoritma Prediksi 2: Weighted Similarity

Bobot = skor *cosine similarity*. Setiap kasus top-k "memilih" labelnya sendiri
dengan bobot sebesar skor similaritynya — label dengan **total bobot tertinggi** menang.
Pendekatan ini lebih sensitif terhadap relevansi kasus dibanding majority vote biasa.

In [58]:
def weighted_similarity(top_k_hasil: List[dict]) -> dict:
    """
    Algoritma Weighted Similarity: bobot = skor similarity.
    Total bobot per label dijumlahkan, label dengan bobot terbesar menang.

    Args:
        top_k_hasil : List[dict] — hasil retrieve_with_scores()

    Returns:
        dict berisi: predicted_label, predicted_label_nama, weight_per_label
    """
    bobot_per_label: Dict[int, float] = {}
    for h in top_k_hasil:
        lbl = h['label']
        bobot_per_label[lbl] = bobot_per_label.get(lbl, 0.0) + h['similarity']

    if not bobot_per_label or sum(bobot_per_label.values()) == 0:
        # Semua similarity = 0 -> tidak ada sinyal yang cukup kuat
        return {
            'predicted_label'      : -1,
            'predicted_label_nama' : 'Tidak Cukup Sinyal',
            'weight_per_label'     : {},
            'metode'               : 'weighted_similarity',
        }

    label_terpilih = max(bobot_per_label, key=bobot_per_label.get)

    return {
        'predicted_label'      : label_terpilih,
        'predicted_label_nama' : LABEL_NAMA.get(label_terpilih, '?'),
        'weight_per_label'     : {k: round(v, 4) for k, v in bobot_per_label.items()},
        'metode'               : 'weighted_similarity',
    }


# ── Test cepat dengan query yang sama ─────────────────────────────────────────
hasil_ws = weighted_similarity(top5_test)

print('Fungsi weighted_similarity() siap!')
print()
print(f'Query              : "{query_test}"')
print(f'Bobot per label    : {hasil_ws["weight_per_label"]}')
print(f'Prediksi           : {hasil_ws["predicted_label_nama"]} (label {hasil_ws["predicted_label"]})')
print()
print('Perbandingan Majority Vote vs Weighted Similarity:')
print(f'  Majority Vote        -> {hasil_mv["predicted_label_nama"]}')
print(f'  Weighted Similarity  -> {hasil_ws["predicted_label_nama"]}')

Fungsi weighted_similarity() siap!

Query              : "tergugat tidak membayar hutang sesuai perjanjian wanprestasi ganti rugi"
Bobot per label    : {0: 0.0865, 1: 0.2486}
Prediksi           : Dikabulkan (label 1)

Perbandingan Majority Vote vs Weighted Similarity:
  Majority Vote        -> Dikabulkan
  Weighted Similarity  -> Dikabulkan


## Cell 6 — Implementasi Fungsi `predict_outcome()`

Sesuai struktur fungsi pada soal:
```python
def predict_outcome(query: str) -> str:
    top_k = retrieve(query, k=5)
    solutions = [case_solutions[c] for c in top_k]
    # Terapkan voting / weighting -> pilih satu ringkasan
    return predicted_solution
```
Diperluas agar dapat memilih metode (`majority` atau `weighted`) dan mengembalikan
detail lengkap (label, solusi teks, top-k case_id) — tetap kompatibel dengan
signature dasar di atas.

In [59]:
def predict_outcome(query: str, k: int = 5, metode: str = 'weighted') -> str:
    """
    Memprediksi solusi/outcome untuk kasus baru berdasarkan top-k kasus termirip.

    Args:
        query  : str  — deskripsi kasus baru
        k      : int  — jumlah kasus termirip yang dipertimbangkan (default 5)
        metode : str  — 'majority' atau 'weighted' (default 'weighted')

    Returns:
        str — ringkasan solusi (amar putusan) dari kasus top-1 yang labelnya
              sama dengan label hasil voting/weighting.
    """
    # 1) Retrieval top-k
    top_k = retrieve(query, k=k)
    top_k_detail = retrieve_with_scores(query, k=k)

    # 2) Ambil solusi tiap kasus top-k dari case_solutions
    solutions = [case_solutions[c] for c in top_k]

    # 3) Terapkan voting / weighting -> pilih label
    if metode == 'majority':
        hasil_prediksi = majority_vote(top_k_detail)
    else:
        hasil_prediksi = weighted_similarity(top_k_detail)

    label_terpilih = hasil_prediksi['predicted_label']

    # 4) Pilih satu ringkasan solusi -> ambil kasus top-1 yang labelnya cocok
    #    dengan hasil voting (representatif), fallback ke top-1 jika tidak ada yang cocok
    predicted_solution = None
    for h in top_k_detail:
        if h['label'] == label_terpilih:
            predicted_solution = case_solutions[h['case_id']]
            break
    if predicted_solution is None:
        predicted_solution = solutions[0] if solutions else 'TIDAK_ADA_SOLUSI'

    return predicted_solution


def predict_outcome_detail(query: str, k: int = 5, metode: str = 'weighted') -> dict:
    """
    Versi predict_outcome() dengan detail lengkap untuk keperluan demo & evaluasi.

    Returns:
        dict — query, top_5_case_ids, predicted_label, predicted_label_nama,
               predicted_solution, metode
    """
    top_k        = retrieve(query, k=k)
    top_k_detail = retrieve_with_scores(query, k=k)
    solutions    = [case_solutions[c] for c in top_k]

    if metode == 'majority':
        hasil_prediksi = majority_vote(top_k_detail)
    else:
        hasil_prediksi = weighted_similarity(top_k_detail)

    label_terpilih = hasil_prediksi['predicted_label']

    predicted_solution = None
    sumber_case_id = None
    for h in top_k_detail:
        if h['label'] == label_terpilih:
            predicted_solution = case_solutions[h['case_id']]
            sumber_case_id = h['case_id']
            break
    if predicted_solution is None:
        predicted_solution = solutions[0] if solutions else 'TIDAK_ADA_SOLUSI'
        sumber_case_id = top_k[0] if top_k else None

    return {
        'query'                 : query,
        'top_5_case_ids'        : top_k,
        'top_5_detail'          : top_k_detail,
        'predicted_label'       : label_terpilih,
        'predicted_label_nama'  : hasil_prediksi['predicted_label_nama'],
        'predicted_solution'    : predicted_solution,
        'sumber_case_id'        : sumber_case_id,
        'metode'                : metode,
    }


print('Fungsi predict_outcome() & predict_outcome_detail() siap!')
print()

# ── Test cepat ──────────────────────────────────────────────────────────────
hasil = predict_outcome_detail(query_test, k=5, metode='weighted')
print(f'Query    : "{hasil["query"]}"')
print(f'Top-5    : {hasil["top_5_case_ids"]}')
print(f'Prediksi : {hasil["predicted_label_nama"]} (dari {hasil["sumber_case_id"]})')
print(f'Solusi   : {hasil["predicted_solution"][:200]}...')

Fungsi predict_outcome() & predict_outcome_detail() siap!

Query    : "tergugat tidak membayar hutang sesuai perjanjian wanprestasi ganti rugi"
Top-5    : ['case_038', 'case_019', 'case_020', 'case_051', 'case_003']
Prediksi : Dikabulkan (dari case_019)
Solusi   : mengadili 1 menyatakan tergugat telah dipanggil dengan patut tetapi tidak hadir 2 mengabulkan gugatan penggugat untuk sebagaian dengan verstek 3 menyatakan sah dan mengikat perjanjian hutang sebagaima...


## Cell 7 — Demo Manual: 5 Contoh Kasus Baru

Sesuai soal poin **d.iv**: siapkan 5 contoh kasus baru, jalankan `predict_outcome()`,
lalu bandingkan dengan putusan sebenarnya (jika query diambil dari kasus yang sudah
ada di case base, sebagai bentuk *sanity check*).

In [60]:
# ── 5 Kasus Baru untuk Demo Manual ────────────────────────────────────────────
# Setiap query mendeskripsikan skenario wanprestasi baru (bukan disalin persis
# dari salah satu dokumen di case base), beserta label_sebenarnya sebagai
# perkiraan/justifikasi manual untuk pembanding pada demo.

demo_kasus_baru = [
    {
        'query_id'        : 'D001',
        'deskripsi'       : 'Kontraktor renovasi rumah tidak menyelesaikan pekerjaan sesuai RAB',
        'query'           : 'tergugat sebagai kontraktor tidak menyelesaikan pekerjaan renovasi rumah sesuai RAB dan perjanjian kerja sehingga penggugat menuntut ganti rugi materiil',
        'label_sebenarnya': 0,  # perkiraan: mirip pola sengketa konstruksi yang sering NO/ditolak (kurang pihak)
    },
    {
        'query_id'        : 'D002',
        'deskripsi'       : 'Debitur tidak membayar cicilan pinjaman sesuai jatuh tempo',
        'query'           : 'penggugat memberikan pinjaman uang kepada tergugat namun tergugat tidak mengembalikan sesuai jatuh tempo yang disepakati dalam perjanjian utang piutang',
        'label_sebenarnya': 1,  # perkiraan: wanprestasi hutang piutang umumnya dikabulkan jika bukti kuat
    },
    {
        'query_id'        : 'D003',
        'deskripsi'       : 'Sengketa kewenangan mengadili — seharusnya diajukan ke PN lain',
        'query'           : 'gugatan diajukan ke pengadilan negeri surabaya namun objek sengketa berada di luar wilayah hukum pengadilan sehingga tidak berwenang mengadili perkara ini',
        'label_sebenarnya': 0,  # kompetensi relatif -> NO
    },
    {
        'query_id'        : 'D004',
        'deskripsi'       : 'Penyewa ruko tidak membayar sewa selama 6 bulan',
        'query'           : 'tergugat sebagai penyewa ruko tidak membayar uang sewa selama enam bulan berturut turut sesuai perjanjian sewa menyewa yang telah disepakati dengan penggugat',
        'label_sebenarnya': 1,
    },
    {
        'query_id'        : 'D005',
        'deskripsi'       : 'Gugatan wanprestasi diajukan sebelum penggugat melunasi kewajibannya',
        'query'           : 'penggugat mengajukan gugatan wanprestasi terhadap tergugat namun penggugat sendiri belum melunasi kewajiban pembayaran sehingga gugatan dinilai prematur',
        'label_sebenarnya': 0,  # prematur -> NO (mirip pola case_002)
    },
]

print(f'{len(demo_kasus_baru)} kasus baru siap untuk demo predict_outcome().')

5 kasus baru siap untuk demo predict_outcome().


In [61]:
print('=' * 78)
print('  DEMO MANUAL — predict_outcome() pada 5 Kasus Baru (metode: weighted)')
print('=' * 78)

demo_results = []

for kasus in demo_kasus_baru:
    hasil = predict_outcome_detail(kasus['query'], k=5, metode='weighted')
    cocok = (hasil['predicted_label'] == kasus['label_sebenarnya'])

    print()
    print(f"[{kasus['query_id']}] {kasus['deskripsi']}")
    print(f"  Query             : {kasus['query'][:90]}...")
    print(f"  Top-5 case_id     : {hasil['top_5_case_ids']}")
    print(f"  Top-5 similarity  : {[h['similarity'] for h in hasil['top_5_detail']]}")
    print(f"  Prediksi label    : {hasil['predicted_label_nama']} (label {hasil['predicted_label']})")
    print(f"  Label sebenarnya  : {LABEL_NAMA.get(kasus['label_sebenarnya'], '?')} (label {kasus['label_sebenarnya']})")
    print(f"  Status            : {'COCOK' if cocok else 'TIDAK COCOK'}")
    print(f"  Solusi (dari {hasil['sumber_case_id']}):")
    print(f"    {hasil['predicted_solution'][:200]}...")

    demo_results.append({
        'query_id'           : kasus['query_id'],
        'deskripsi'          : kasus['deskripsi'],
        'query'              : kasus['query'],
        'top_5_case_ids'     : hasil['top_5_case_ids'],
        'predicted_label'    : hasil['predicted_label'],
        'predicted_label_nama': hasil['predicted_label_nama'],
        'label_sebenarnya'   : kasus['label_sebenarnya'],
        'cocok'              : cocok,
        'predicted_solution' : hasil['predicted_solution'],
        'sumber_case_id'     : hasil['sumber_case_id'],
    })

print()
print('=' * 78)
n_cocok = sum(d['cocok'] for d in demo_results)
print(f'  Akurasi Demo Manual: {n_cocok}/{len(demo_results)} ({n_cocok/len(demo_results)*100:.0f}%)')
print('=' * 78)

  DEMO MANUAL — predict_outcome() pada 5 Kasus Baru (metode: weighted)

[D001] Kontraktor renovasi rumah tidak menyelesaikan pekerjaan sesuai RAB
  Query             : tergugat sebagai kontraktor tidak menyelesaikan pekerjaan renovasi rumah sesuai RAB dan pe...
  Top-5 case_id     : ['case_002', 'case_036', 'case_044', 'case_004', 'case_038']
  Top-5 similarity  : [0.1674, 0.1178, 0.0688, 0.0642, 0.0526]
  Prediksi label    : Ditolak/NO (label 0)
  Label sebenarnya  : Ditolak/NO (label 0)
  Status            : COCOK
  Solusi (dari case_002):
    mengadili dalam konpensi dalam eksepsi menerima eksepsi dari tergugat dalam pokok perkara menyatakan gugatan penggugat tidak dapat diterima niet ontvankelijkeverklaard dalam rekonpensi menyatakan guga...

[D002] Debitur tidak membayar cicilan pinjaman sesuai jatuh tempo
  Query             : penggugat memberikan pinjaman uang kepada tergugat namun tergugat tidak mengembalikan sesu...
  Top-5 case_id     : ['case_039', 'case_022', 'case_026', 'c

## Cell 7b — Perbandingan Majority Vote vs Weighted Similarity (5 Kasus)

Menjalankan kedua metode secara berdampingan untuk melihat apakah keduanya
menghasilkan prediksi yang sama atau berbeda.

In [62]:
print('=' * 78)
print('  PERBANDINGAN METODE: Majority Vote vs Weighted Similarity')
print('=' * 78)
print(f'{"Query":<6} {"Majority Vote":<20} {"Weighted Sim.":<20} {"Sama?"}')
print('-' * 60)

for kasus in demo_kasus_baru:
    hasil_mv_demo = predict_outcome_detail(kasus['query'], k=5, metode='majority')
    hasil_ws_demo = predict_outcome_detail(kasus['query'], k=5, metode='weighted')
    sama = 'Ya' if hasil_mv_demo['predicted_label'] == hasil_ws_demo['predicted_label'] else 'Tidak'
    print(f"{kasus['query_id']:<6} {hasil_mv_demo['predicted_label_nama']:<20} {hasil_ws_demo['predicted_label_nama']:<20} {sama}")

print('=' * 78)

  PERBANDINGAN METODE: Majority Vote vs Weighted Similarity
Query  Majority Vote        Weighted Sim.        Sama?
------------------------------------------------------------
D001   Ditolak/NO           Ditolak/NO           Ya
D002   Dikabulkan           Dikabulkan           Ya
D003   Ditolak/NO           Ditolak/NO           Ya
D004   Ditolak/NO           Ditolak/NO           Ya
D005   Ditolak/NO           Ditolak/NO           Ya


## Cell 8 — Simpan `data/results/predictions.csv`

Sesuai soal poin **d.v**: simpan hasil prediksi dengan kolom
`query_id | predicted_solution | top_5_case_ids`.

In [63]:
# ── Susun DataFrame predictions.csv sesuai format soal ────────────────────────
rows_predictions = []
for d in demo_results:
    rows_predictions.append({
        'query_id'           : d['query_id'],
        'query'              : d['query'],
        'predicted_solution' : d['predicted_solution'],
        'top_5_case_ids'     : ', '.join(d['top_5_case_ids']),
        'predicted_label'    : d['predicted_label'],
        'predicted_label_nama': d['predicted_label_nama'],
        'label_sebenarnya'   : d['label_sebenarnya'],
        'cocok'              : d['cocok'],
    })

df_predictions = pd.DataFrame(rows_predictions)
df_predictions.to_csv(PATH_PREDICTIONS, index=False, encoding='utf-8-sig')

print('=' * 62)
print('  PREDICTIONS.CSV TERSIMPAN')
print('=' * 62)
print(f'  Path  : {PATH_PREDICTIONS}')
print(f'  Baris : {len(df_predictions)}')
print(f'  Kolom : {list(df_predictions.columns)}')
print('=' * 62)
print()
print('Preview (kolom utama sesuai soal):')
df_predictions[['query_id', 'predicted_solution', 'top_5_case_ids']].head()

  PREDICTIONS.CSV TERSIMPAN
  Path  : data/results/predictions.csv
  Baris : 5
  Kolom : ['query_id', 'query', 'predicted_solution', 'top_5_case_ids', 'predicted_label', 'predicted_label_nama', 'label_sebenarnya', 'cocok']

Preview (kolom utama sesuai soal):


,query_id,predicted_solution,top_5_case_ids
0,D001,mengadili dalam konpensi dalam eksepsi menerim...,"case_002, case_036, case_044, case_004, case_038"
1,D002,mengadili 1 mengabulkan gugatan penggugat untu...,"case_039, case_022, case_026, case_051, case_008"
2,D003,mengadilidalam eksepsi menerima eksepsi dari t...,"case_014, case_042, case_045, case_047, case_054"
3,D004,mengadili dalam eksepsi menerima eksepsi tergu...,"case_008, case_042, case_012, case_002, case_041"
4,D005,mengadili perkara dalam gugatan ini menghukum ...,"case_021, case_042, case_039, case_002, case_025"


## Cell 9 — Ringkasan Akhir Tahap 4

In [64]:
print()
print('=' * 62)
print('  RINGKASAN AKHIR TAHAP 4 — CASE SOLUTION REUSE')
print('=' * 62)
print(f'  Case base (case_solutions)   : {len(case_solutions)} kasus')
print(f'  Algoritma prediksi           : Majority Vote & Weighted Similarity')
print(f'  k (top-k retrieval)          : 5')
print()
print(f'  Demo Manual (5 kasus baru):')
print(f'    Akurasi vs label perkiraan : {n_cocok}/{len(demo_results)} ({n_cocok/len(demo_results)*100:.0f}%)')
print()
print('  Output Tahap 4:')
ada = '[OK]' if os.path.exists(PATH_PREDICTIONS) else '[GAGAL]'
print(f'    {ada} {PATH_PREDICTIONS}')
print(f'    [OK] Fungsi predict_outcome(query, k=5, metode="weighted"/"majority")')
print(f'    [OK] Struktur case_solutions = {{case_id: solusi_text}}')
print('=' * 62)
print('  STATUS: TAHAP 4 SELESAI! Lanjut ke Tahap 5 (Model Evaluation).')
print('=' * 62)


  RINGKASAN AKHIR TAHAP 4 — CASE SOLUTION REUSE
  Case base (case_solutions)   : 54 kasus
  Algoritma prediksi           : Majority Vote & Weighted Similarity
  k (top-k retrieval)          : 5

  Demo Manual (5 kasus baru):
    Akurasi vs label perkiraan : 4/5 (80%)

  Output Tahap 4:
    [OK] data/results/predictions.csv
    [OK] Fungsi predict_outcome(query, k=5, metode="weighted"/"majority")
    [OK] Struktur case_solutions = {case_id: solusi_text}
  STATUS: TAHAP 4 SELESAI! Lanjut ke Tahap 5 (Model Evaluation).
